## Data Availability Notice

The data used in this notebook was provided by **Dogs Trust** under a formal data sharing agreement as part of an MSc Data Science dissertation at Liverpool John Moores University, supervised by Dr Jon Bielby.

**The raw data files are not included in this repository.** To reproduce the analysis, the following data extracts from the Dogs Trust Salesforce CRM are required:

- `Dog Affiliation and Status_Sub Status History Pull.xlsx` (Affiliation History sheet)
- `Sub Status History v2.csv`

Requests for data access should be directed to the Dogs Trust Data Science & Analytics team.

All data handling was conducted in accordance with GDPR and Dogs Trust's data governance protocols.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
sub_status_v2 = pd.read_csv('../data/[Dogs Trust data file - not included in repo]')  # adjust filename to match exactly

print("Shape:", sub_status_v2.shape)
print("\nColumns:", sub_status_v2.columns.tolist())
print("\nUnique fields tracked:")
print(sub_status_v2['field'].unique() if 'field' in sub_status_v2.columns else "no 'field' column - check column names")

In [ ]:
print(sub_status_v2['new_value'].value_counts().head(20))

In [ ]:
# Convert edit_date to datetime and sort
sub_status_v2['edit_date'] = pd.to_datetime(sub_status_v2['edit_date'])
sub_status_v2 = sub_status_v2.sort_values(['dog_id', 'edit_date'])

# Calculate duration in each sub-status (time until next status change for that dog)
sub_status_v2['next_edit_date'] = sub_status_v2.groupby('dog_id')['edit_date'].shift(-1)
sub_status_v2['duration_days'] = (
    sub_status_v2['next_edit_date'] - sub_status_v2['edit_date']
).dt.days

# Focus on the hold-related sub-statuses
hold_statuses = ['On Hold: Vet', 'On Hold: Behaviour', 'Under Assessment', 'Reserved']

hold_analysis = sub_status_v2[
    sub_status_v2['new_value'].isin(hold_statuses)
].groupby('new_value')['duration_days'].agg(['count', 'mean', 'median', 'sum']).round(1)

print("Duration Analysis by Hold/Status Type:")
print(hold_analysis)

In [ ]:
fig_holds = go.Figure()

hold_analysis_sorted = hold_analysis.sort_values('mean', ascending=False).reset_index()

fig_holds.add_trace(go.Bar(
    x=hold_analysis_sorted['new_value'],
    y=hold_analysis_sorted['mean'],
    name='Mean Duration (days)',
    marker_color='steelblue',
    text=hold_analysis_sorted['mean'],
    textposition='outside'
))

fig_holds.add_trace(go.Scatter(
    x=hold_analysis_sorted['new_value'],
    y=hold_analysis_sorted['sum'],
    name='Total Days Consumed',
    mode='lines+markers+text',
    text=hold_analysis_sorted['sum'].apply(lambda x: f'{int(x):,}'),
    textposition='top center',
    textfont=dict(size=11, color='black'),
    marker=dict(size=8, color='red'),
    line=dict(color='red', width=2),
    yaxis='y2'
))

fig_holds.update_layout(
    title='RQ1 — Bottleneck Causes: Per-Dog Duration vs System-Wide Impact',
    xaxis_title='Hold / Status Type',
    yaxis_title='Mean Duration per Dog (days)',
    yaxis2=dict(title='Total Days Consumed (all dogs)', overlaying='y', side='right'),
    height=500,
    width=950,
    legend=dict(x=0.6, y=1.15)
)

fig_holds.write_image('../outputs/rq1_bottleneck_causes.png', width=1100, height=600, scale=2)
fig_holds.write_html('../outputs/rq1_bottleneck_causes.html')


fig_holds.show()

In [ ]:
# Load affiliation history to get LOS per dog
affiliation_history = pd.read_excel(
    '../data/[Dogs Trust data file - not included in repo]',
    sheet_name='Affiliation History'
)

# Calculate total LOS per dog 
intake_dates = affiliation_history[affiliation_history['Process'] == 'Intake']\
    .sort_values('Start Date')\
    .groupby('Dog ID')['Start Date'].first().reset_index()
intake_dates.columns = ['Dog ID', 'Intake Date']

last_records = affiliation_history.sort_values('Start Date')\
    .groupby('Dog ID').last().reset_index()
last_records['Exit Date'] = last_records['End Date'].fillna(last_records['Start Date'])

los_df = intake_dates.merge(last_records[['Dog ID', 'Exit Date']], on='Dog ID')
los_df['Length of Stay'] = (
    pd.to_datetime(los_df['Exit Date']) - pd.to_datetime(los_df['Intake Date'])
).dt.days

print(f"LOS calculated for {len(los_df)} dogs")
print(los_df['Length of Stay'].describe().round(1))

In [ ]:
# Flag dogs that experienced each hold type
hold_types = ['On Hold: Vet', 'On Hold: Behaviour']

# Get set of dog IDs for each hold type
dogs_with_holds = {}
for hold in hold_types:
    dogs_with_holds[hold] = set(
        sub_status_v2[sub_status_v2['new_value'] == hold]['dog_id'].unique()
    )
    print(f"{hold}: {len(dogs_with_holds[hold])} dogs")

# Check ID format compatibility between the two tables
print("\nSample Dog ID from affiliation (los_df):", los_df['Dog ID'].iloc[0], type(los_df['Dog ID'].iloc[0]))
print("Sample dog_id from sub_status_v2:", sub_status_v2['dog_id'].iloc[0], type(sub_status_v2['dog_id'].iloc[0]))

In [ ]:
# Add hold flags to los_df
los_df['Had Vet Hold'] = los_df['Dog ID'].isin(dogs_with_holds['On Hold: Vet'])
los_df['Had Behaviour Hold'] = los_df['Dog ID'].isin(dogs_with_holds['On Hold: Behaviour'])
los_df['Had Any Hold'] = los_df['Had Vet Hold'] | los_df['Had Behaviour Hold']

# Compare mean LOS
print("Average Length of Stay Comparison:\n")
print(f"Dogs with NO hold:        {los_df[~los_df['Had Any Hold']]['Length of Stay'].mean():.1f} days")
print(f"Dogs with a VET hold:     {los_df[los_df['Had Vet Hold']]['Length of Stay'].mean():.1f} days")
print(f"Dogs with a BEHAVIOUR hold: {los_df[los_df['Had Behaviour Hold']]['Length of Stay'].mean():.1f} days")
print(f"Dogs with ANY hold:       {los_df[los_df['Had Any Hold']]['Length of Stay'].mean():.1f} days")

# Also show how many dogs
print(f"\nDog counts:")
print(f"No hold: {(~los_df['Had Any Hold']).sum()}")
print(f"Any hold: {los_df['Had Any Hold'].sum()}")

In [ ]:
import plotly.graph_objects as go

categories = ['No Hold', 'Vet Hold', 'Behaviour Hold', 'Any Hold']
values = [
    los_df[~los_df['Had Any Hold']]['Length of Stay'].mean(),
    los_df[los_df['Had Vet Hold']]['Length of Stay'].mean(),
    los_df[los_df['Had Behaviour Hold']]['Length of Stay'].mean(),
    los_df[los_df['Had Any Hold']]['Length of Stay'].mean(),
]

fig_link = go.Figure()
fig_link.add_trace(go.Bar(
    x=categories,
    y=values,
    text=[f'{v:.0f} days' for v in values],
    textposition='outside',
    marker_color=['#8FBC8F', '#FFA500', '#DC143C', '#4169E1']
))

fig_link.update_layout(
    title='How Holds Affect Length of Stay — Dogs on Hold Stay Much Longer',
    xaxis_title='Dog Category',
    yaxis_title='Average Length of Stay (days)',
    height=500,
    width=850,
    showlegend=False
)

fig_link.write_image('../outputs/rq1_holds_vs_los.png', width=1000, height=600, scale=2)
fig_link.write_html('../outputs/rq1_holds_vs_los.html')

fig_link.show()